# Step 3: dataset and protocol audit

Development-only validation for the Essenza thesis. No fitting, prediction, or final-test array deserialization. Historical test diagnostics are explicitly identified as prior evidence.

The code cells run sequentially with the original project's Python interpreter via `execute_audit.py`. No IPython magics or interactive-kernel state are required.

## Context and methods

Grain: one canonical, single-fragment molecule with 25 binary aroma annotations. Unrecorded annotations are treated as negatives, not experimentally confirmed absence. Dataset version and feature schema are frozen. The 1,320-row test partition remains unchanged.

### Key assumptions

The historical source audit is usable only if its dataset ID and the original manifest hashes still match. Scope excludes refetching sources, re-extracting every feature, retraining, and examining test labels. Source timeliness means a fixed revision, not the latest online dataset.

In [1]:
import os, sys, json, time
from pathlib import Path
os.environ["ESSENZA_THREADS"] = "1"
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[name] = "1"
ROOT = Path.cwd().resolve()
assert ROOT == Path(r"C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier")
sys.path.insert(0, str(ROOT))
import numpy as np
import scipy.sparse as sp
from sklearn.model_selection import GroupKFold
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from src.artifacts import digest, json_hash
from src.featurize import FeatureSpec
from src.splits import development_partitions, split_indices, group_split_indices, scaffold_groups
from src.config import load_config
DATA = ROOT / "data/builds/audited-20260904-v2"
OUT = ROOT / "reports/step3_20260908"
SEEDS = [42, 123, 2026]
started = time.monotonic()
manifest = json.loads((DATA / "dataset_manifest.json").read_text(encoding="utf-8"))
previous = json.loads((ROOT / "reports/audit_20260904/dataset_audit.json").read_text())
print({"scope": "development only", "seeds": SEEDS})

{'scope': 'development only', 'seeds': [42, 123, 2026]}


## Data

### Verify byte integrity and feature contract

Test-related files are read only as byte streams for SHA-256. Their arrays and SMILES are not parsed. Snapshot checks verify provenance; no source data is downloaded.

In [2]:
reference = json.loads((ROOT / "reports/step0_20260907/artifact_reconciliation.json").read_text(encoding="utf-8-sig"))
verified = []
for group in reference["copies"]:
    for item in group["files"]:
        path = Path(group["destination"]) / item["path"]
        assert digest(path) == item["sha256"], str(path)
        verified.append(str(path.relative_to(ROOT)))
for name, expected in manifest["files"].items():
    assert Path(name).name == name
    assert digest(DATA / name) == expected, name
assert manifest["dataset_id"] == json_hash(manifest["files"]) == previous["dataset_id"]
spec = FeatureSpec.from_dict(manifest["feature_spec"])
assert spec.schema_id == manifest["feature_schema_id"] == previous["feature_schema_id"]
assert spec.to_dict() == json.loads((DATA / "feature_spec.json").read_text())
assert manifest["labels"] == json.loads((DATA / "label_names.json").read_text())
assert manifest["protocol_version"] == 2 and manifest["identity_join_validated"]
outer_train, outer_test = map(set, (manifest["split"]["train"], manifest["split"]["test"]))
assert not outer_train & outer_test
assert outer_train | outer_test == set(range(manifest["n_train"] + manifest["n_test"]))
print({"artifacts_verified": len(verified), "dataset_id": manifest["dataset_id"],
       "feature_schema_id": spec.schema_id, "features": spec.n_features,
       "test_arrays_deserialized": False})

{'artifacts_verified': 29, 'dataset_id': '5c0b0e7c9a56eb60fbc344fc3b6629a160dfa723fdb7b6ba398e621870f2f558', 'feature_schema_id': 'a1dce64f7591f9b40f5f2332d805b7f1c7f91f24d9af0dcd0964cda7c231c018', 'features': 2053, 'test_arrays_deserialized': False}


### Inspect development features and labels

Only `X_train.npz`, `Y_train.npz`, and development SMILES are parsed. CSR data remains sparse for feature checks. Descriptor summaries and prevalence refer exclusively to development.

In [3]:
X = sp.load_npz(DATA / "X_train.npz").tocsr()
Y = sp.load_npz(DATA / "Y_train.npz").toarray()
smiles = (DATA / "smiles_train.txt").read_text(encoding="utf-8").splitlines()
assert X.shape == (5383, 2053) and Y.shape == (5383, 25)
assert X.dtype == np.float32 and Y.dtype == np.uint8
assert np.isfinite(X.data).all() and np.isin(Y, [0, 1]).all()
assert len(smiles) == len(set(smiles)) == len(Y) and all(smiles)
assert not any("." in value for value in smiles)
bits = X[:, :spec.n_bits]
assert np.isin(bits.data, [0, 1]).all()
counts = Y.sum(axis=0).astype(int)
assert counts.tolist() == manifest["train_positives"]
assert dict(zip(manifest["labels"], counts.tolist())) == previous["train_positives"]
descriptors = X[:, spec.n_bits:].toarray()
assert np.all(descriptors[:, [0, 2, 3, 4]] >= 0)
descriptor_stats = {name: {"min": float(descriptors[:,j].min()), "max":float(descriptors[:,j].max())}
    for j, name in enumerate(["MolWt", "MolLogP", "NumHDonors", "NumHAcceptors", "TPSA"])}
profile = {"development_rows":len(Y), "features":X.shape[1], "labels":Y.shape[1],
    "nonfinite_feature_values":0, "duplicate_development_smiles":0,
    "rows_without_target_labels":int((Y.sum(axis=1) == 0).sum()),
    "positive_counts":dict(zip(manifest["labels"], counts.tolist())),
    "bit_density":bits.count_nonzero() / (bits.shape[0] * bits.shape[1]),
    "descriptor_ranges":descriptor_stats}
print(json.dumps(profile, indent=2))

{
  "development_rows": 5383,
  "features": 2053,
  "labels": 25,
  "nonfinite_feature_values": 0,
  "duplicate_development_smiles": 0,
  "rows_without_target_labels": 381,
  "positive_counts": {
    "apple": 309,
    "balsamic": 399,
    "citrus": 396,
    "earthy": 374,
    "ethereal": 332,
    "fatty": 610,
    "floral": 1360,
    "fresh": 458,
    "fruity": 1942,
    "green": 1593,
    "herbal": 986,
    "meaty": 258,
    "mint": 267,
    "musty": 233,
    "nutty": 342,
    "oily": 532,
    "rose": 394,
    "spicy": 509,
    "sulfurous": 329,
    "sweet": 1259,
    "tropical": 320,
    "vegetable": 262,
    "waxy": 454,
    "winey": 238,
    "woody": 821
  },
  "bit_density": 0.011334057304244844,
  "descriptor_ranges": {
    "MolWt": {
      "min": 4.002999782562256,
      "max": 1297.1280517578125
    },
    "MolLogP": {
      "min": -17.406400680541992,
      "max": 17.71430015563965
    },
    "NumHDonors": {
      "min": 0.0,
      "max": 24.0
    },
    "NumHAcceptors": {
   

## Results

### Verify the primary partitions and class support

Regenerate development partitions with seed 42 and compare with the archived split indices. The archived run is read only; no experiment is initialized. Require disjoint roles, score-fold coverage, an isolated threshold holdout, and both classes for all labels.

In [4]:
cfg = load_config(ROOT / "config.safe-training.yaml")
split_cfg = {k:v for k,v in cfg["split"].items() if k != "test_size"}
primary = development_partitions(Y, seed=42, **split_cfg)
archived = json.loads((ROOT / "runs/_reference/audited-20260904-v2/run.json").read_text())
assert primary == archived["splits"]
fit = np.array(primary["fit"])
threshold = set(primary["threshold"])
assert set(fit) | threshold == set(range(len(Y))) and not set(fit) & threshold
def fold_summary(folds, group_values=None):
    summaries = []
    score_indices = []
    for number, fold in enumerate(folds):
        roles = {role:set(fold[role]) for role in ["train", "stop", "score"]}
        assert set.union(*roles.values()) == set(fit)
        assert not any(roles[a] & roles[b] for a,b in [("train","stop"),("train","score"),("stop","score")])
        assert not any(indices & threshold for indices in roles.values())
        missing = {}
        for role, indices in roles.items():
            positives = Y[sorted(indices)].sum(axis=0)
            missing[role] = [manifest["labels"][j] for j in range(Y.shape[1])
                             if positives[j] in [0, len(indices)]]
        if group_values is not None:
            grouped = {role:set(group_values[sorted(indices)]) for role,indices in roles.items()}
            assert not any(grouped[a] & grouped[b] for a,b in [("train","stop"),("train","score"),("stop","score")])
        score_indices += fold["score"]
        summaries.append({"fold":number, "sizes":{k:len(v) for k,v in roles.items()}, "missing_class_labels":missing})
    assert sorted(score_indices) == sorted(fit.tolist())
    return summaries
primary_summary = fold_summary(primary["folds"])
assert not any(labels for fold in primary_summary for labels in fold["missing_class_labels"].values())
threshold_counts = Y[primary["threshold"]].sum(axis=0)
assert np.all((threshold_counts > 0) & (threshold_counts < len(threshold)))
print({"fit_rows":len(fit), "threshold_rows":len(threshold), "folds":primary_summary,
       "matches_archived_indices":True, "split_sha256":json_hash(primary)})

{'fit_rows': 4575, 'threshold_rows': 808, 'folds': [{'fold': 0, 'sizes': {'train': 2592, 'stop': 458, 'score': 1525}, 'missing_class_labels': {'train': [], 'stop': [], 'score': []}}, {'fold': 1, 'sizes': {'train': 2592, 'stop': 458, 'score': 1525}, 'missing_class_labels': {'train': [], 'stop': [], 'score': []}}, {'fold': 2, 'sizes': {'train': 2592, 'stop': 458, 'score': 1525}, 'missing_class_labels': {'train': [], 'stop': [], 'score': []}}], 'matches_archived_indices': True, 'split_sha256': '92cdc377dcfc37d5c1f930769d4fbe510326f7c0fcaf905e9ec213c69e0c810e'}


### Freeze development-only sensitivity partitions

Seeds 123 and 2026 change CV/early-stop partitions and learner/resampling randomness while retaining the seed-42 outer development/test boundary, 25 labels, features, and 808-row threshold holdout. The scaffold comparison uses seed 42 and groups only the primary fitting pool. No new outer dataset is built.

These checks measure partition feasibility, not model scores. Missing classes are reported without searching for a more favorable seed or dropping a label.

In [5]:
groups = scaffold_groups(smiles)
unique_groups, sizes = np.unique(groups, return_counts=True)
def sensitivity_folds(seed, scaffold=False):
    cv = GroupKFold(n_splits=3, shuffle=True, random_state=seed) if scaffold else MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    result = []
    for number, (train_local, score_local) in enumerate(cv.split(np.zeros((len(fit),1)), Y[fit], groups[fit] if scaffold else None)):
        train_pool = fit[train_local]
        if scaffold:
            inner, stop = group_split_indices(Y[train_pool], groups[train_pool], .15, seed + number + 1)
        else:
            inner, stop = split_indices(Y[train_pool], .15, seed + number + 1)
        result.append({"train":train_pool[inner].tolist(), "stop":train_pool[stop].tolist(), "score":fit[score_local].tolist()})
    return result
assert sensitivity_folds(42) == primary["folds"]
sensitivity = {}
sensitivity_summary = {}
for seed in [123, 2026]:
    name = f"random_seed_{seed}"
    folds = sensitivity_folds(seed)
    sensitivity[name] = {"seed":seed, "grouped":False, "folds":folds}
    sensitivity_summary[name] = fold_summary(folds)
folds = sensitivity_folds(42, scaffold=True)
sensitivity["scaffold_seed_42"] = {"seed":42, "grouped":True, "folds":folds}
sensitivity_summary["scaffold_seed_42"] = fold_summary(folds, groups)
group_profile = {"development_scaffold_groups":len(unique_groups),
    "largest_group_rows":int(sizes.max()), "acyclic_rows":int(np.sum(groups == "")),
    "group_definition":"Murcko scaffold with chirality retained; empty scaffold remains one group",
    "scope":"development only; threshold excluded from all sensitivity fitting/scoring",
    "historical_test_scaffold_overlap":{"rows":previous["test_rows_with_scaffold_seen_in_train"], "denominator":previous["n_test"], "recomputed":False}}
print(json.dumps({"scaffolds":group_profile,"sensitivity":sensitivity_summary}, indent=2))

{
  "scaffolds": {
    "development_scaffold_groups": 606,
    "largest_group_rows": 2303,
    "acyclic_rows": 2303,
    "group_definition": "Murcko scaffold with chirality retained; empty scaffold remains one group",
    "scope": "development only; threshold excluded from all sensitivity fitting/scoring",
    "historical_test_scaffold_overlap": {
      "rows": 1231,
      "denominator": 1320,
      "recomputed": false
    }
  },
  "sensitivity": {
    "random_seed_123": [
      {
        "fold": 0,
        "sizes": {
          "train": 2592,
          "stop": 458,
          "score": 1525
        },
        "missing_class_labels": {
          "train": [],
          "stop": [],
          "score": []
        }
      },
      {
        "fold": 1,
        "sizes": {
          "train": 2592,
          "stop": 458,
          "score": 1525
        },
        "missing_class_labels": {
          "train": [],
          "stop": [],
          "score": []
        }
      },
      {
        "fold": 

### Save inspectable audit evidence

All outputs are under this Step 3 evidence directory. Frozen split files are not operational runs. The two 4-hour Optuna budgets come from the user's explicit choice and do not authorize training.

In [6]:
source_rows = {}
for name, values in manifest["source_audit"].items():
    source_rows[name] = {**values, "accepted_fraction":values["accepted"]/values["rows"],
        "valid_identity_without_mapped_labels":values["rows"]-values["accepted"]-values["unmapped_identity"]-values["invalid_or_multifragment"]}
audit = {
    "status":"PASS_DATASET_INTEGRITY", "dataset_id":manifest["dataset_id"],
    "dataset_manifest_sha256":digest(DATA/"dataset_manifest.json"),
    "feature_schema_id":spec.schema_id, "source_revision":manifest["source_revision"],
    "artifact_hash_checks":len(verified), "profile":profile, "primary_partitions":primary_summary,
    "fit_rows":len(fit), "threshold_rows":len(threshold), "primary_split_sha256":json_hash(primary),
    "sensitivity_partitions":sensitivity_summary, "scaffold_profile":group_profile,
    "source_record_audit":source_rows,
    "historical_identity_overlap":{"count":previous["identity_overlap"], "recomputed":False},
    "label_missingness":manifest["label_missingness"], "evaluation_status":manifest["evaluation_status"],
    "test_arrays_deserialized":False, "test_smiles_parsed":False, "training_started":False,
    "audit_compute_seconds":time.monotonic()-started}
payloads = {"dataset_audit.json":audit, "development_partitions.json":primary,
            "sensitivity_partitions.json":sensitivity, "development_scaffold_groups.json":groups.tolist()}
for name, payload in payloads.items():
    path = OUT/name
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        if name != "dataset_audit.json":
            assert old == payload, f"Existing frozen output differs: {name}"
        else:
            assert old["dataset_id"] == payload["dataset_id"]
            continue
    else:
        path.write_text(json.dumps(payload,indent=2)+"\n",encoding="utf-8")
print({"saved":list(payloads),"training_started":False,"test_arrays_deserialized":False})

{'saved': ['dataset_audit.json', 'development_partitions.json', 'sensitivity_partitions.json', 'development_scaffold_groups.json'], 'training_started': False, 'test_arrays_deserialized': False}


### Resolve scaffold feasibility before any model fitting

The initial grouped split lacks both classes for apple and winey in one early-stopping holdout. The frozen sensitivity design therefore uses fixed primary-recipe boosting rounds and no sensitivity early-stopping holdout. Each old train+stop set is joined into training, keeping score groups isolated. Apply exactly the same policy to random seeds 42, 123, and 2026; the seed-42 control must be fitted anew under this policy. No label is dropped and no seed is searched.

All sensitivities are conditional diagnostics of the primary selected recipe, not a fresh nested-CV estimate.

In [7]:
fixed_variants = {"random_seed_42":{"seed":42,"grouped":False,"folds":primary["folds"]}, **sensitivity}
execution_partitions = {}
execution_summary = {}
for name, value in fixed_variants.items():
    converted = []
    for fold in value["folds"]:
        train_indices = sorted(set(fold["train"]) | set(fold["stop"]))
        score_indices = fold["score"]
        assert not set(train_indices) & set(score_indices)
        assert set(train_indices) | set(score_indices) == set(fit)
        assert not (set(train_indices) | set(score_indices)) & threshold
        for indices in [train_indices, score_indices]:
            positives = Y[indices].sum(axis=0)
            assert np.all((positives > 0) & (positives < len(indices)))
        if value["grouped"]:
            assert not set(groups[train_indices]) & set(groups[score_indices])
        converted.append({"train":train_indices, "score":score_indices})
    execution_partitions[name] = {"seed":value["seed"],"grouped":value["grouped"],"folds":converted}
    execution_summary[name] = [{"train":len(f["train"]), "score":len(f["score"])} for f in converted]
resolution = {
    "status":"FEASIBLE_FIXED_RECIPE_SENSITIVITY",
    "reason":"Original grouped early-stop fold 1 lacks class support for apple/winey",
    "rounds":"per-label median best_iterations from the primary selected trial, frozen before sensitivity",
    "parameters":"same primary selected recipe for all variants; no retuning",
    "threshold_holdout_reused":False, "test_access":False,
    "missing_train_or_score_classes":0, "partition_sizes":execution_summary,
    "limitations":"Conditional sensitivity, not unbiased nested CV; primary selection previously used the fitting pool"}
extreme_indices = np.r_[np.argsort(descriptors[:,0])[:5], np.argsort(descriptors[:,0])[-3:]]
domain_examples = [{"development_index":int(i),"smiles":smiles[i],"MolWt":float(descriptors[i,0]),
    "labels":[manifest["labels"][j] for j in np.flatnonzero(Y[i])]} for i in extreme_indices]
for name, value in {"sensitivity_execution_partitions.json":execution_partitions,
                    "sensitivity_resolution.json":resolution,
                    "development_domain_examples.json":domain_examples}.items():
    path = OUT/name
    if path.exists():
        assert json.loads(path.read_text(encoding="utf-8")) == value
    else:
        path.write_text(json.dumps(value,indent=2)+"\n",encoding="utf-8")
print(json.dumps(resolution,indent=2))

{
  "status": "FEASIBLE_FIXED_RECIPE_SENSITIVITY",
  "reason": "Original grouped early-stop fold 1 lacks class support for apple/winey",
  "rounds": "per-label median best_iterations from the primary selected trial, frozen before sensitivity",
  "parameters": "same primary selected recipe for all variants; no retuning",
  "threshold_holdout_reused": false,
  "test_access": false,
  "missing_train_or_score_classes": 0,
  "partition_sizes": {
    "random_seed_42": [
      {
        "train": 3050,
        "score": 1525
      },
      {
        "train": 3050,
        "score": 1525
      },
      {
        "train": 3050,
        "score": 1525
      }
    ],
    "random_seed_123": [
      {
        "train": 3050,
        "score": 1525
      },
      {
        "train": 3050,
        "score": 1525
      },
      {
        "train": 3050,
        "score": 1525
      }
    ],
    "random_seed_2026": [
      {
        "train": 3050,
        "score": 1525
      },
      {
        "train": 3050,
   

## Takeaways

Review the executed integrity, class-support, and grouping outputs before interpreting the dataset. Zero target annotations do not establish true absence. Source acceptance is a record-level measure, not the number of distinct molecules contributed by a source. Test overlap diagnostics come from the unchanged historical audit. Grouped sensitivity is conditional on the selected primary recipe and does not constitute an external or nested-CV performance estimate.

The accompanying protocol records the decisions and limitations; no model performance is produced here.